# Sinhala Speech-to-Text: Gradio Demo

Record or upload Sinhala audio and get a transcription from [`aadhilimam/whisper-small-sinhala`](https://www.kaggle.com/models/aadhilimam/whisper-small-sinhala), a `whisper-small` model fine-tuned on the OpenSLR SLR52 corpus.

### How to try it
1. Click **Copy & Edit** (or **Edit**, if this is your notebook).
2. In the session settings, make sure **Internet** is on. Kaggle may ask you to verify your phone number first.
3. Click **Run All**.
4. Open the `gradio.live` link printed by the last cell.

The demo only works while your notebook session is running, and the link is temporary. It runs on CPU by default, so turn on a GPU in the settings if you want faster transcription.

Audio you submit goes to your own Kaggle session, through Gradio's temporary share link. Anyone with the link can use the demo while it is running.

### What to expect
The model scored 9.13% CER and 36.47% WER on clips from speakers it never heard, but most of those test sentences also appear in its training data, so new sentences may do worse. It writes Sinhala script only, with no digits, Latin text or punctuation. See the [model card](https://www.kaggle.com/models/aadhilimam/whisper-small-sinhala) for details.

In [ ]:
!pip install -q "transformers>=4.46,<5" "gradio>=5,<6" librosa soundfile remotezip

In [ ]:
import glob
import os

os.environ["USE_TF"] = "0"

import gradio as gr
import librosa
import torch
from gradio_client import Client, handle_file
from remotezip import RemoteZip
from transformers import pipeline

configs = glob.glob("/kaggle/input/**/config.json", recursive=True)
if not configs:
    raise FileNotFoundError("Model not found. Add aadhilimam/whisper-small-sinhala as an input to this notebook.")
MODEL_DIR = os.path.dirname(configs[0])

device = 0 if torch.cuda.is_available() else -1
print("Model:", MODEL_DIR)
print("Running on:", "GPU" if device == 0 else "CPU")
asr = pipeline("automatic-speech-recognition", model=MODEL_DIR, device=device, chunk_length_s=30)

In [ ]:
def transcribe(audio_path):
    if not audio_path:
        return "Please record or upload some audio first."
    audio, _ = librosa.load(audio_path, sr=16000, mono=True)
    result = asr({"raw": audio, "sampling_rate": 16000},
                 generate_kwargs={"language": "sinhala", "task": "transcribe"})
    return result["text"].strip()

In [ ]:
# Example clips from held-out test speakers, fetched from OpenSLR SLR52 (CC BY-SA 4.0).
EXAMPLE_REFERENCES = {
    "8b97848022": "ඇලෙක්සාර් රෑන්කින් කියන්නෙ හොදම එක නෙමෙයි",
    "9753dab5f9": "එක්තරා උදෑසනක ට පමණ කරන ලද ස්කෑන් එකකින්",
    "e41cb38d3f": "ඔබට දැන් ඉතාම නිවැරැදි පිළිතුරක් ලැබී තිබෙනවා",
}


def fetch_example(clip_id):
    url = f"https://www.openslr.org/resources/52/asr_sinhala_{clip_id[0]}.zip"
    with RemoteZip(url) as archive:
        name = next(n for n in archive.namelist() if n.endswith(f"/{clip_id}.flac"))
        return archive.extract(name, "/kaggle/working/examples")


examples = {}
for clip_id in EXAMPLE_REFERENCES:
    try:
        examples[clip_id] = fetch_example(clip_id)
    except Exception as exc:
        print(f"Could not fetch example {clip_id}: {exc}")
print(f"Fetched {len(examples)} of {len(EXAMPLE_REFERENCES)} example clips")

In [ ]:
demo = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath", label="Sinhala audio"),
    outputs=gr.Textbox(label="Transcription", lines=4),
    examples=[[path] for path in examples.values()] or None,
    title="Sinhala Speech-to-Text",
    description="Whisper-small fine-tuned on OpenSLR SLR52. Record or upload Sinhala speech. "
                "Output is Sinhala script only, with no digits, Latin text or punctuation.",
    flagging_mode="never",
)

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    # A saved, headless run: check the demo works end to end, then shut it down.
    _, local_url, share_url = demo.launch(share=True, prevent_thread_lock=True, inline=False)
    print("Share link created:", bool(share_url))
    client = Client(share_url or local_url)
    for clip_id, path in examples.items():
        print("CLIP:", clip_id)
        print("REF: ", EXAMPLE_REFERENCES[clip_id])
        print("HYP: ", client.predict(handle_file(path), api_name="/predict"))
    demo.close()
    print("DEMO CHECK PASSED")
else:
    demo.launch(share=True)